# Revenue Forecast Notebook

This notebook builds a simple revenue forecasting workflow for the NorthRiver Analytics project.

## Goals
- Load monthly KPI data from `../data/monthly_financial_kpis.csv`
- Build a baseline rolling-average forecast
- Build an improved linear-trend forecast using NumPy
- Evaluate model behavior with MAE and MAPE
- Export structured outputs back to `../data/`


<p>
FILE: revenue_forecast.py <br>
PROJECT: Financial Risk Analytics Dashboard – NorthRiver Analytics <br>
FOLDER: python <br>
AUTHOR: Anthony Washington <br>
DATE: April 2026

PURPOSE:
  This script builds a simple revenue forecasting model using historical
  monthly revenue data from the NorthRiver Analytics B2B SaaS simulation.

  It is designed to demonstrate core quantitative analyst skills:
    - Loading and preparing time-series data
    - Building and evaluating a baseline forecast model (rolling average)
    - Improving the model using a linear trend approach
    - Documenting assumptions and model behavior
    - Exporting structured outputs for reporting

WHY THIS MATTERS FOR A QUANTITATIVE ANALYST ROLE:
  Revenue forecasting is one of the most common tasks for a quant analyst.
  The ability to build a simple, explainable model — evaluate how well it
  works, understand where it fails, and document your assumptions — is more
  valuable than knowing complex algorithms. This script shows exactly that.

INPUTS: <br>
  ../data/monthly_financial_kpis.csv

OUTPUTS: <br>
  ../data/revenue_forecast_6mo.csv       — 6-month forward forecast <br>
  ../data/forecast_model_evaluation.csv  — model accuracy on held-out data
<p>


SECTION 1: IMPORTS


WHY: Python cannot do everything on its own. We import external libraries
that give us tools to work with data, math, and dates.

  pandas  — the most important data library in Python for analysts.
            Think of it like Excel in code form. It handles tables (called
            DataFrames), lets you filter rows, create columns, sort data,
            and export CSVs. Used in almost every data analyst role.

  numpy   — short for "Numerical Python." Handles math operations on arrays
            of numbers very efficiently. polyfit() and polyval() (used below)
            come from NumPy. Critical for any quantitative or analytics role.

  datetime — Python's built-in module for working with dates. We use it here
             to build future time periods for our forecast.


In [33]:
import pandas as pd
import numpy as np
from datetime import datetime


SECTION 2: LOAD AND PREPARE THE DATA


WHY: Before you can analyze anything, you need to load the raw data and make
sure it is in the right shape. This is called "data preparation" or "wrangling"
and is a major part of any real analyst job.

WHAT WE ARE DOING:
  - Reading the CSV file from the data/ folder into a pandas DataFrame
  - Assigning column names manually because our CSV has no header row
  - Creating a "period" column that combines year and month into a real date
    so pandas can sort and work with time properly
  - Sorting by period (oldest to newest) so the time series is in order
  - Resetting the index so row numbers go 0, 1, 2, 3... after sorting


Read the CSV. The file has no header row so we provide column names manually.
WHY names=[...]: Without this, pandas would treat row 1 as the header, which
is wrong — row 1 is actual data, not labels.


In [26]:
df = pd.read_csv(
    "../data/monthly_financial_kpis.csv",
    names=["year", "month", "month_name", "revenue", "cost", "margin", "margin_pct"]
)

# Create a real date column from year + month.
# WHY: Pandas needs a proper datetime column to understand time order.
# pd.to_datetime() converts a string like "2023-3" into a real date: 2023-03-01.
# format="%Y-%m" tells pandas exactly how to read it (4-digit year, 2-digit month).
df["period"] = pd.to_datetime(
    df["year"].astype(str) + "-" + df["month"].astype(str),
    format="%Y-%m"
)

# Sort by period ascending (oldest month first) and reset the row index.
# WHY sort: Our rolling average and trend model both depend on data being in
# chronological order. If months are out of order, the math will be wrong.
# WHY reset_index(drop=True): After sorting, pandas keeps the old row numbers.
# drop=True throws them away and renumbers rows 0, 1, 2, 3... cleanly.
df = df.sort_values("period").reset_index(drop=True)

# Quick data check — print the first and last few rows so you can verify it loaded correctly.
# WHY: Always visually inspect your data after loading. This catches issues like
# wrong column names, missing values, or bad date parsing early.
print("=== DATA LOADED ===")
print(f"Total months of data: {len(df)}")
print(f"Date range: {df['period'].min().strftime('%b %Y')} to {df['period'].max().strftime('%b %Y')}")
print(f"Total revenue across all periods: ${df['revenue'].sum():,.0f}")
print()


=== DATA LOADED ===
Total months of data: 37
Date range: Mar 2023 to Mar 2026
Total revenue across all periods: $930,714



SECTION 3: TRAIN / TEST SPLIT


WHY: In forecasting and modeling, you never test your model on the same data
you used to build it. That would be like giving students the answer key and
then grading them on the same test — it tells you nothing useful.

WHAT WE ARE DOING:
  - "Train" data = all months before October 2025. This is what we use to
    build (train) the model.
  - "Test" data = last 6 months (Oct 2025 – Mar 2026). This is data the model
    has NOT seen, so we can check how well it would have predicted real results.

WHY 6 MONTHS:
  For a business planning model, 6 months of holdout gives a realistic
  picture of near-term forecast accuracy — exactly what a quant analyst
  would want to show to leadership.


Split into training and test sets.
WHY "2025-10-01": Everything before October 2025 is training data.
October 2025 onward is our holdout (test) set.


In [27]:
train = df[df["period"] < "2025-10-01"].copy()
test  = df[df["period"] >= "2025-10-01"].copy()

print("=== TRAIN / TEST SPLIT ===")
print(f"Training months: {len(train)}  ({train['period'].min().strftime('%b %Y')} to {train['period'].max().strftime('%b %Y')})")
print(f"Test months:     {len(test)}   ({test['period'].min().strftime('%b %Y')} to {test['period'].max().strftime('%b %Y')})")
print()


=== TRAIN / TEST SPLIT ===
Training months: 31  (Mar 2023 to Sep 2025)
Test months:     6   (Oct 2025 to Mar 2026)



SECTION 4: MODEL V1 — ROLLING 3-MONTH AVERAGE (BASELINE)


WHY A ROLLING AVERAGE:
  The simplest, most explainable forecast you can build. A rolling average
  says: "My best guess for next month is the average of the last 3 months."
  It is the kind of model you can explain to any non-technical stakeholder
  in one sentence, which is exactly what a quant analyst needs to be able
  to do.

WHAT IS A ROLLING AVERAGE:
  For each row, look back at the previous N rows and take their average.
  window=3 means "look at the 3 most recent months."
  .shift(1) means "shift the result forward by 1 row" — this prevents
  the model from using the current month's actual data to predict itself
  (called "data leakage" — a serious modeling mistake).

EXAMPLE:
  If Jan=$30k, Feb=$38k, Mar=$42k → rolling average for April = (30+38+42)/3 = $36.7k
  The .shift(1) ensures we use the 3 months BEFORE the prediction month.

ASSUMPTION DOCUMENTED:
  This model assumes the near-term future will look like a simple average
  of the recent past. It ignores growth trends and seasonality. That is a
  known limitation we will address in Model V2 below.


Compute rolling 3-month average on TRAINING data only, then shift by 1.
WHY on train only: We do not want to "leak" test data into the model.


In [28]:
rolling_mean = train["revenue"].rolling(window=3).mean()

# The last value of the rolling mean from training is our single forecast
# value for all test months in this simple baseline model.
# WHY: A basic rolling average does not project forward month-by-month;
# it just gives you a single "expected value" based on recent history.
baseline_forecast = rolling_mean.iloc[-1]

# Apply that single value to all test rows.
test["forecast_v1"] = baseline_forecast

print("=== MODEL V1: ROLLING 3-MONTH AVERAGE ===")
print(f"Baseline forecast value (avg of last 3 training months): ${baseline_forecast:,.0f}")
print()


=== MODEL V1: ROLLING 3-MONTH AVERAGE ===
Baseline forecast value (avg of last 3 training months): $36,897



SECTION 5: EVALUATE MODEL V1 — HOW WELL DID IT DO?


WHY EVALUATE:
  Building a model is only half the job. A quantitative analyst must also
  understand how their model behaves — where it is accurate, where it misses,
  and why. This is what separates a real analyst from someone who just runs
  code without understanding results.

METRICS WE ARE USING:
  error         = actual revenue - forecasted revenue
                  Positive error = model underestimated (actual was higher)
                  Negative error = model overestimated (actual was lower)

  abs_error     = absolute value of error (removes the +/- sign so we can
                  compare magnitude of misses fairly)

  pct_error     = error / actual revenue
                  Shows how large the miss was relative to actual revenue.
                  A $5k miss on $10k revenue is 50% off — very bad.
                  A $5k miss on $100k revenue is 5% off — pretty good.

  MAE           = Mean Absolute Error = average of all abs_errors
                  The single most useful summary metric for forecast accuracy.
                  "On average, we were off by $X per month."

  MAPE          = Mean Absolute Percentage Error = average of all pct_errors
                  "On average, we were off by X% per month."
                  Industry standard for business forecast evaluation.


In [29]:
test["error"]     = test["revenue"] - test["forecast_v1"]
test["abs_error"] = test["error"].abs()
test["pct_error"] = test["error"] / test["revenue"]

mae_v1  = test["abs_error"].mean()
mape_v1 = test["pct_error"].abs().mean()

print("=== MODEL V1 EVALUATION ===")
print(test[["period", "revenue", "forecast_v1", "error", "pct_error"]].to_string(index=False))
print()
print(f"Mean Absolute Error (MAE):  ${mae_v1:,.0f}")
print(f"Mean Absolute % Error (MAPE): {mape_v1:.1%}")
print()
print("INTERPRETATION:")
print("  - Positive errors = actual revenue was higher than forecast (model underestimated)")
print("  - Negative errors = actual revenue was lower than forecast (model overestimated)")
print("  - A MAPE above 20% suggests the model is missing a structural pattern")
print("    such as a growth trend, seasonality, or a business shift.")
print()


=== MODEL V1 EVALUATION ===
    period  revenue  forecast_v1         error  pct_error
2025-10-01  43906.0 36896.666667   7009.333333   0.159644
2025-11-01  41250.0 36896.666667   4353.333333   0.105535
2025-12-01  44761.0 36896.666667   7864.333333   0.175696
2026-01-01  42625.0 36896.666667   5728.333333   0.134389
2026-02-01  38500.0 36896.666667   1603.333333   0.041645
2026-03-01  16500.0 36896.666667 -20396.666667  -1.236162

Mean Absolute Error (MAE):  $7,826
Mean Absolute % Error (MAPE): 30.9%

INTERPRETATION:
  - Positive errors = actual revenue was higher than forecast (model underestimated)
  - Negative errors = actual revenue was lower than forecast (model overestimated)
  - A MAPE above 20% suggests the model is missing a structural pattern
    such as a growth trend, seasonality, or a business shift.



SECTION 6: MODEL V2 — LINEAR TREND (IMPROVED FORECAST)


WHY IMPROVE THE MODEL:
  Model V1 (rolling average) treats every month the same — it does not
  "know" that revenue has been growing over time. If the business is
  growing, a flat average will always underestimate the future.

  Model V2 fits a straight line through all the historical revenue data.
  The line captures the overall growth trend and projects it forward.
  This is called "linear regression" — one of the most fundamental tools
  in quantitative analysis, statistics, and finance.

HOW np.polyfit WORKS:
  polyfit(x, y, deg=1) finds the best-fit line through the data points.
  x = row numbers (0, 1, 2, 3... representing each month in order)
  y = actual revenue values
  deg=1 = degree 1 polynomial, which is just a straight line: y = mx + b
  Returns [m, b] where m = slope (monthly growth rate) and b = starting value.

HOW np.polyval WORKS:
  polyval(coeffs, x) takes the line equation [m, b] and calculates y for
  any new x value. We use it to project revenue forward for future months.

ASSUMPTION DOCUMENTED:
  This model assumes the historical linear growth rate continues unchanged.
  It does not account for seasonality, market changes, or business events.
  These are documented limitations — a quant analyst always documents what
  their model does NOT capture.


x = integer index for each month (0 = first month, 1 = second, etc.)
y = actual revenue values


In [30]:
x = np.arange(len(train))
y = train["revenue"].values

# Fit a degree-1 polynomial (straight line) through the training data.
# coeffs[0] = slope (how much revenue changes per month on average)
# coeffs[1] = intercept (starting value when x=0)
coeffs = np.polyfit(x, y, deg=1)

print("=== MODEL V2: LINEAR TREND ===")
print(f"Trend slope (monthly revenue change): ${coeffs[0]:,.0f} per month")
print(f"Trend intercept (starting value):     ${coeffs[1]:,.0f}")
print()

# Evaluate Model V2 on the test set.
# WHY: We use the same test months as V1 so we can compare them fairly.
test_x = np.arange(len(train), len(train) + len(test))
test["forecast_v2"] = np.polyval(coeffs, test_x)

test["error_v2"]     = test["revenue"] - test["forecast_v2"]
test["abs_error_v2"] = test["error_v2"].abs()
test["pct_error_v2"] = test["error_v2"] / test["revenue"]

mae_v2  = test["abs_error_v2"].mean()
mape_v2 = test["pct_error_v2"].abs().mean()

print("=== MODEL V2 EVALUATION ===")
print(test[["period", "revenue", "forecast_v2", "error_v2", "pct_error_v2"]].to_string(index=False))
print()
print(f"Mean Absolute Error (MAE):    ${mae_v2:,.0f}")
print(f"Mean Absolute % Error (MAPE):  {mape_v2:.1%}")
print()
print("MODEL COMPARISON:")
print(f"  V1 Rolling Average MAE:  ${mae_v1:,.0f}  MAPE: {mape_v1:.1%}")
print(f"  V2 Linear Trend    MAE:  ${mae_v2:,.0f}  MAPE: {mape_v2:.1%}")
print()


=== MODEL V2: LINEAR TREND ===
Trend slope (monthly revenue change): $1,111 per month
Trend intercept (starting value):     $6,021

=== MODEL V2 EVALUATION ===
    period  revenue  forecast_v2      error_v2  pct_error_v2
2025-10-01  43906.0 40455.412903   3450.587097      0.078590
2025-11-01  41250.0 41566.190726   -316.190726     -0.007665
2025-12-01  44761.0 42676.968548   2084.031452      0.046559
2026-01-01  42625.0 43787.746371  -1162.746371     -0.027279
2026-02-01  38500.0 44898.524194  -6398.524194     -0.166195
2026-03-01  16500.0 46009.302016 -29509.302016     -1.788443

Mean Absolute Error (MAE):    $7,154
Mean Absolute % Error (MAPE):  35.2%

MODEL COMPARISON:
  V1 Rolling Average MAE:  $7,826  MAPE: 30.9%
  V2 Linear Trend    MAE:  $7,154  MAPE: 35.2%



SECTION 7: FORWARD FORECAST — NEXT 6 MONTHS


WHY:
  The test set tells us how our model would have done on past data.
  The forward forecast is what actually matters to business stakeholders —
  what do we expect revenue to look like in the next 6 months?

WHAT WE ARE DOING:
  - Building a date range of the next 6 months after our last data point
  - Using Model V2 (linear trend) to project revenue for each of those months
  - Saving results to a CSV so they can be used in Power BI or Excel reporting

WHY V2 OVER V1:
  If V2 showed a lower MAPE than V1 in the evaluation step, it is the
  better model for this dataset. If they are similar, V1 is simpler to
  explain. In an interview, you should be able to explain why you chose one.


Last date in our dataset + 1 month = first forecast month


In [31]:
last_date = df["period"].max()

# Generate 6 future month-start dates using pandas date_range.
# freq="MS" = "Month Start" — generates the 1st of each month.
# periods=7 gives us 7 dates; we skip the first one (which is last_date itself)
# so we get 6 true future months.
future_periods = pd.date_range(last_date, periods=7, freq="MS")[1:]

# The x-values for future months continue counting from where training ended.
future_x = np.arange(len(df), len(df) + 6)

# Apply Model V2 to get the forecast values.
future_forecast = np.polyval(coeffs, future_x)

# Build a clean output DataFrame.
forecast_df = pd.DataFrame({
    "period":       future_periods.strftime("%b %Y"),
    "forecast_revenue": np.round(future_forecast, 2)
})

print("=== 6-MONTH FORWARD FORECAST (Model V2: Linear Trend) ===")
print(forecast_df.to_string(index=False))
print()
print("ASSUMPTION NOTE:")
print("  This forecast assumes the historical monthly growth trend continues unchanged.")
print("  It does not account for seasonality, churn events, or market shifts.")
print("  Results should be reviewed alongside customer pipeline and retention data.")
print()


=== 6-MONTH FORWARD FORECAST (Model V2: Linear Trend) ===
  period  forecast_revenue
Apr 2026          47120.08
May 2026          48230.86
Jun 2026          49341.64
Jul 2026          50452.41
Aug 2026          51563.19
Sep 2026          52673.97

ASSUMPTION NOTE:
  This forecast assumes the historical monthly growth trend continues unchanged.
  It does not account for seasonality, churn events, or market shifts.
  Results should be reviewed alongside customer pipeline and retention data.



SECTION 8: EXPORT OUTPUTS


WHY EXPORT:
  Analysts do not just run scripts — they produce outputs that other people
  (business leaders, Power BI dashboards, Excel reports) can use. Saving CSVs
  is the standard handoff format between SQL/Python work and reporting tools.

WHY THESE TWO FILES:
  - revenue_forecast_6mo.csv: the forward-looking forecast for stakeholders.
    Goes into Power BI or Excel as a reporting table.
  - forecast_model_evaluation.csv: documents how the model performed on
    historical data. This is your audit trail and proof of analytical rigor.


Save 6-month forward forecast.


In [32]:
forecast_df.to_csv("../data/revenue_forecast_6mo.csv", index=False)
print("Saved: ../data/revenue_forecast_6mo.csv")

# Save model evaluation (test set comparison of actual vs V1 and V2).
eval_cols = ["period", "revenue", "forecast_v1", "error", "pct_error",
             "forecast_v2", "error_v2", "pct_error_v2"]
test[eval_cols].to_csv("../data/forecast_model_evaluation.csv", index=False)
print("Saved: ../data/forecast_model_evaluation.csv")

print()
print("=== SCRIPT COMPLETE ===")
print("Next steps:")
print("  1. Review forecast_model_evaluation.csv — which model performed better?")
print("  2. Load revenue_forecast_6mo.csv into Power BI or Excel for visualization.")
print("  3. Document findings in docs/CASE_STUDY_QUANT_ANALYST.md")


# =============================================================================
# SUMMARY OF CONCEPTS USED IN THIS SCRIPT
# (Review these before interviews)
#
#   pandas DataFrame     — table of data in Python (like a spreadsheet in code)
#   pd.read_csv()        — loads a CSV file into a DataFrame
#   pd.to_datetime()     — converts text into a real date Python understands
#   .sort_values()       — sorts rows by a column (like sorting in Excel)
#   .rolling(window=3)   — computes a rolling/moving window calculation
#   .mean()              — calculates the arithmetic average
#   .shift(1)            — shifts values forward/backward to avoid data leakage
#   Train/test split     — dividing data into "model building" and "checking" sets
#   np.polyfit(x,y,deg)  — fits a polynomial (line, curve) to data points
#   np.polyval(c,x)      — evaluates a polynomial at new x values (project forward)
#   MAE                  — average dollar error per prediction
#   MAPE                 — average % error per prediction (most common in business)
#   Data leakage         — accidentally using future data to predict the past (bad)
#   Forward forecast     — predicting future values the model has not seen
# =============================================================================


Saved: ../data/revenue_forecast_6mo.csv
Saved: ../data/forecast_model_evaluation.csv

=== SCRIPT COMPLETE ===
Next steps:
  1. Review forecast_model_evaluation.csv — which model performed better?
  2. Load revenue_forecast_6mo.csv into Power BI or Excel for visualization.
  3. Document findings in docs/CASE_STUDY_QUANT_ANALYST.md
